# Second notebook

Fits the two models, M4 and M7

# Part 1 — Setup

In [ ]:
!pip install -q "pymc>=5.10" arviz

import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pymc as pm, arviz as az, pytensor.tensor as pt
import warnings, json, os, time, gc, sys
warnings.filterwarnings("ignore")
az.style.use("arviz-whitegrid")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "savefig.bbox": "tight", "font.size": 9})
os.makedirs("figures", exist_ok=True)

SEED = 1

# Warmup is long on purpose. The seven sites differ in precision by a factor of
# about 350 (Mongolia's within-site SD is 3.08, the Philippines' is 1041), so the
# sampler needs many iterations to adapt a mass matrix that suits both ends. The
# supplied R code uses n_warmup = 1e4 for the same reason.
DRAWS, TUNE, CHAINS = 1000, 2000, 2

# Two chains rather than four, to halve the wall clock. Split-Rhat still works:
# it divides each chain in half, so two chains give four sequences to compare.
# Four is better and is what a final run should use; two is valid and is enough
# to detect a failure to converge. Set CHAINS = 4 if time allows.

MODELS_TO_FIT = ['M4 hier correlated', 'M7 uniform priors']   # this machine fits only these
LL_THIN  = 8     # thinning for the log-likelihood, keeps 500 draws
PPC_DRAW = 150   # replicate datasets per model

FITS, LOOD, RESULT = {}, {}, {}
BLUE, RED, GREY, PURPLE = "#2B5D9B", "#B03A2E", "#8A8A8A", "#7B4FA8"
q = lambda x: np.percentile(x, [2.5, 97.5])

print("PyMC", pm.__version__, "| ArviZ", az.__version__)

PyMC 5.28.5 | ArviZ 0.22.0


# Part 2 — Data import and preparation

Upload `microcredit_profit.csv`. One row per household: study site, treatment
assignment, and business profit in USD PPP per fortnight.

In [ ]:
from google.colab import files
up = files.upload()

df = pd.read_csv([f for f in up if f.endswith('.csv')][0])
SITES = ["Mexico","Mongolia","Bosnia","India","Morocco","Philippines","Ethiopia"]
K = len(SITES)

if "site_idx" not in df.columns:
    df["site_idx"] = pd.Categorical(df["site"], categories=SITES).codes
df["site_idx"] = df["site_idx"].astype(int)
df["treat"]    = df["treat"].astype(int)
df["site"]     = [SITES[i] for i in df.site_idx]

n_before = len(df)
df = df[["site","site_idx","treat","y"]].dropna().reset_index(drop=True)
assert df.site_idx.nunique() == K,      "expected seven sites"
assert set(df.treat.unique()) == {0,1}, "treat must be 0 or 1"
assert len(df) == n_before,             "unexpected missing values"

idx   = df.site_idx.to_numpy()
treat = df.treat.to_numpy().astype(float)
y     = df.y.to_numpy()
N     = len(y)

print(f"{N:,} households across {K} sites, no missing values")
df.head()

Saving microcredit_profit.csv to microcredit_profit.csv
35,303 households across 7 sites, no missing values


,site,site_idx,treat,y
0,Mexico,0,0,0.000000
1,Mexico,0,1,513.246492
2,Mexico,0,1,0.000000
3,Mexico,0,0,0.000000
4,Mexico,0,0,0.000000


# Part 3 — Exploratory analysis

Two questions before any model is fitted. What shape does the outcome have, and
how much do the sites differ? The answers determine which modelling choices
later need defending.

In [ ]:
rows = []
for j, s in enumerate(SITES):
    g  = df[df.site_idx == j]
    yc = g.loc[g.treat==0,"y"].to_numpy(); yt = g.loc[g.treat==1,"y"].to_numpy()
    n0, n1 = len(yc), len(yt)
    s2 = ((n1-1)*yt.var(ddof=1) + (n0-1)*yc.var(ddof=1))/(n1+n0-2)
    gy = g.y.to_numpy(); nz = gy[gy != 0]
    rows.append({"site": s, "N": n0+n1, "treated": n1, "control": n0,
                 "control mean": yc.mean(), "difference": yt.mean()-yc.mean(),
                 "std error": np.sqrt(s2*(1/n1+1/n0)), "within SD": np.sqrt(s2),
                 "neg %": 100*(gy<0).mean(), "zero %": 100*(gy==0).mean(),
                 "pos %": 100*(gy>0).mean(),
                 "mean if active": nz.mean(), "SD if active": nz.std()})
desc = pd.DataFrame(rows).set_index("site")

n_neg, n_zero, n_pos = int((y<0).sum()), int((y==0).sum()), int((y>0).sum())
skew = float(((y-y.mean())**3).mean()/y.std()**3)
kurt = float(((y-y.mean())**4).mean()/y.var()**2)

print(f"POOLED   N={N:,}   mean={y.mean():.2f}   SD={y.std():.1f}   median={np.median(y):.1f}")
print(f"         coefficient of variation = {y.std()/y.mean():.1f}\n")
print("The outcome has three distinct parts, not two:")
print(f"   negative  {n_neg:6,}  ({100*n_neg/N:5.1f}%)   businesses making a loss, down to {y.min():,.0f}")
print(f"   zero      {n_zero:6,}  ({100*n_zero/N:5.1f}%)   exactly zero, a genuine point mass")
print(f"   positive  {n_pos:6,}  ({100*n_pos/N:5.1f}%)   up to {y.max():,.0f}\n")
print(f"skewness = {skew:.2f}, so the heavy tail is on the LEFT: the extreme values")
print(f"are large losses, not large profits.")
print(f"kurtosis = {kurt:.0f}, against 3 for a normal.\n")
print(f"within-site SD  {desc['within SD'].min():8.2f} to {desc['within SD'].max():9.2f}"
      f"   (factor of {desc['within SD'].max()/desc['within SD'].min():.0f})")
print(f"std error       {desc['std error'].min():8.2f} to {desc['std error'].max():9.2f}"
      f"   (factor of {desc['std error'].max()/desc['std error'].min():.0f})")
print(f"raw estimate    {desc['difference'].min():8.2f} to {desc['difference'].max():9.2f}")
print(f"share at zero   {desc['zero %'].min():8.1f}% to {desc['zero %'].max():8.1f}%\n")
print("Four features that shape every modelling choice below:")
print("  a majority of outcomes are exactly zero, which no continuous density can match")
print("  a tenth are negative, so the outcome cannot be logged and is not a count")
print("  the sites differ enormously in precision, so pooling will act unevenly")
print("  the tails are heavy and left leaning, so the normal likelihood is an assumption\n")

RESULT["data"] = {"n": int(N), "mean": float(y.mean()), "sd": float(y.std()),
                  "n_negative": n_neg, "n_zero": n_zero, "n_positive": n_pos,
                  "pct_zero": float(100*n_zero/N), "skewness": skew, "kurtosis": kurt,
                  "per_site": desc.round(3).to_dict("index")}
desc.round(2)

POOLED   N=35,303   mean=46.18   SD=435.5   median=0.0
         coefficient of variation = 9.4

The outcome has three distinct parts, not two:
   negative   3,493  (  9.9%)   businesses making a loss, down to -40,854
   zero      21,928  ( 62.1%)   exactly zero, a genuine point mass
   positive   9,882  ( 28.0%)   up to 20,173

skewness = -12.96, so the heavy tail is on the LEFT: the extreme values
are large losses, not large profits.
kurtosis = 2699, against 3 for a normal.

within-site SD      3.07 to   1039.75   (factor of 339)
std error           0.22 to     78.13   (factor of 351)
raw estimate       -4.55 to     66.56
share at zero       10.6% to     82.7%

Four features that shape every modelling choice below:
  a majority of outcomes are exactly zero, which no continuous density can match
  a tenth are negative, so the outcome cannot be logged and is not a count
  the sites differ enormously in precision, so pooling will act unevenly
  the tails are heavy and left leaning, so th

,N,treated,control,control mean,difference,std error,within SD,neg %,zero %,pos %,mean if active,SD if active
site,,,,,,,,,,,,
Mexico,16560,8262,8298,14.38,-4.55,5.88,378.26,4.19,82.72,13.09,70.06,907.62
Mongolia,961,701,260,-0.68,-0.34,0.22,3.07,17.59,77.63,4.79,-4.14,5.37
Bosnia,1195,627,568,97.34,37.53,19.78,341.48,0.00,72.55,27.45,426.40,541.74
India,6863,3599,3264,29.37,16.72,11.83,489.43,4.66,74.63,20.71,150.36,963.02
Morocco,5498,2720,2778,81.05,17.54,11.40,422.67,27.59,18.48,53.93,110.07,465.75
Philippines,1113,892,221,381.35,66.56,78.13,1039.75,0.90,13.39,85.71,501.89,1101.38
Ethiopia,3113,1586,1527,13.00,7.29,7.89,220.16,25.15,10.60,64.25,18.70,232.72


# Part 5 — Model specifications


In [ ]:
S0  = np.full(K, 300.0)
MU0 = np.full(K, 50.)
TA0 = np.full(K, 5.)

def M0():
    """complete pooling: one shared treatment effect"""
    with pm.Model(coords={"site":SITES}) as m:
        mu_k = pm.Normal("mu_k", 0, 1000, dims="site", initval=MU0)
        tau  = pm.Normal("tau", 0, 1000, initval=5.)
        sk   = pm.HalfNormal("sigma_k", 1000, dims="site", initval=S0)
        pm.Normal("y", mu_k[idx] + tau*treat, sk[idx], observed=y)
    return m

def M1():
    """no pooling: each site estimated alone"""
    with pm.Model(coords={"site":SITES}) as m:
        mu_k  = pm.Normal("mu_k", 0, 1000, dims="site", initval=MU0)
        tau_k = pm.Normal("tau_k", 0, 1000, dims="site", initval=TA0)
        sk    = pm.HalfNormal("sigma_k", 1000, dims="site", initval=S0)
        pm.Normal("y", mu_k[idx] + tau_k[idx]*treat, sk[idx], observed=y)
    return m

def M2():
    """hierarchical on the treatment effects only"""
    with pm.Model(coords={"site":SITES}) as m:
        mu_k = pm.Normal("mu_k", 0, 1000, dims="site", initval=MU0)
        tau  = pm.Normal("tau", 0, 1000, initval=5.)
        st   = pm.HalfNormal("sigma_tau", 50, initval=10.)
        tau_k= pm.Normal("tau_k", tau, st, dims="site", initval=TA0)
        sk   = pm.HalfNormal("sigma_k", 1000, dims="site", initval=S0)
        pm.Normal("y", mu_k[idx] + tau_k[idx]*treat, sk[idx], observed=y)
        pm.Normal("tau_new", tau, st)
    return m

def M3():
    """hierarchical on baselines and effects -- main specification"""
    with pm.Model(coords={"site":SITES}) as m:
        mu = pm.Normal("mu", 0, 1000, initval=50.)
        tau= pm.Normal("tau", 0, 1000, initval=5.)
        sm = pm.HalfNormal("sigma_mu", 500, initval=100.)
        st = pm.HalfNormal("sigma_tau", 50, initval=10.)
        mu_k  = pm.Normal("mu_k",  mu,  sm, dims="site", initval=MU0)
        tau_k = pm.Normal("tau_k", tau, st, dims="site", initval=TA0)
        sk = pm.HalfNormal("sigma_k", 1000, dims="site", initval=S0)
        pm.Normal("y", mu_k[idx] + tau_k[idx]*treat, sk[idx], observed=y)
        pm.Normal("tau_new", tau, st)
    return m

def M4():
    """baselines and effects correlated across sites (Meager's joint model)"""
    with pm.Model(coords={"site":SITES}) as m:
        mu = pm.Normal("mu", 0, 1000, initval=50.)
        tau= pm.Normal("tau", 0, 1000, initval=5.)
        chol, corr, stds = pm.LKJCholeskyCov(
            "chol", n=2, eta=2,
            sd_dist=pm.HalfNormal.dist(sigma=np.array([500., 50.]), shape=2),
            compute_corr=True)
        theta = pm.MvNormal("theta_k", mu=pt.stack([mu, tau]), chol=chol,
                            shape=(K, 2), initval=np.column_stack([MU0, TA0]))
        mu_k  = pm.Deterministic("mu_k",  theta[:,0], dims="site")
        tau_k = pm.Deterministic("tau_k", theta[:,1], dims="site")
        pm.Deterministic("rho", corr[0,1])
        st = pm.Deterministic("sigma_tau", stds[1])
        sk = pm.HalfNormal("sigma_k", 1000, dims="site", initval=S0)
        pm.Normal("y", mu_k[idx] + tau_k[idx]*treat, sk[idx], observed=y)
        pm.Normal("tau_new", tau, st)
    return m

def M5():
    """as M3, with the residual scales pooled on the log scale"""
    with pm.Model(coords={"site":SITES}) as m:
        mu = pm.Normal("mu", 0, 1000, initval=50.)
        tau= pm.Normal("tau", 0, 1000, initval=5.)
        sm = pm.HalfNormal("sigma_mu", 500, initval=100.)
        st = pm.HalfNormal("sigma_tau", 50, initval=10.)
        mu_k  = pm.Normal("mu_k",  mu,  sm, dims="site", initval=MU0)
        tau_k = pm.Normal("tau_k", tau, st, dims="site", initval=TA0)
        lm = pm.Normal("log_s_mu", 5, 3, initval=5.5)
        ls = pm.HalfNormal("log_s_sd", 2, initval=1.)
        log_sk = pm.Normal("log_sigma_k", lm, ls, dims="site", initval=np.log(S0))
        sk = pm.Deterministic("sigma_k", pt.exp(log_sk), dims="site")
        pm.Normal("y", mu_k[idx] + tau_k[idx]*treat, sk[idx], observed=y)
        pm.Normal("tau_new", tau, st)
    return m

def M7():
    """as M3 with the uniform priors of the supplied Stan code"""
    with pm.Model(coords={"site":SITES}) as m:
        mu = pm.Normal("mu", 0, 1000, initval=50.)
        tau= pm.Normal("tau", 0, 1000, initval=5.)
        sm = pm.Uniform("sigma_mu", 0, 1e5, initval=100.)
        st = pm.Uniform("sigma_tau", 0, 1e5, initval=10.)
        mu_k  = pm.Normal("mu_k",  mu,  sm, dims="site", initval=MU0)
        tau_k = pm.Normal("tau_k", tau, st, dims="site", initval=TA0)
        sk = pm.Uniform("sigma_k", 0, 1e5, dims="site", initval=S0)
        pm.Normal("y", mu_k[idx] + tau_k[idx]*treat, sk[idx], observed=y)
        pm.Normal("tau_new", tau, st)
    return m

BUILDERS = {"M0 complete pooling":M0, "M1 no pooling":M1, "M2 hier tau only":M2,
            "M3 hier both":M3, "M4 hier correlated":M4, "M5 hier + sigma":M5,
            "M7 uniform priors":M7}

# M6 is defined above but held back from this run. It is the only model here with
# a non-normal outcome, and a Student t with nu fixed at 4 is being asked to fit
# tails far heavier than it expects, which makes it much slower than the rest.
# To include it:  BUILDERS["M6 Student t"] = M6
print(f"{len(BUILDERS)} models will be fitted; M6 defined but deferred")

7 models will be fitted; M6 defined but deferred


# Fitting — M4 and M7 only

In [ ]:
import os
SAVE_DIR = "/content/fits"; os.makedirs(SAVE_DIR, exist_ok=True)
_p = lambda n,k: os.path.join(SAVE_DIR, f"{n.replace(' ','_')}_{k}.nc")

for name in list(BUILDERS):
    if name not in MODELS_TO_FIT: BUILDERS.pop(name)
print("this notebook fits:", ", ".join(BUILDERS), "\n")

for name, build in BUILDERS.items():
    fit_p, loo_p = _p(name,"fit"), _p(name,"loo")
    if name in FITS:
        print(f"  {name:<22} in memory, skipped"); continue
    if os.path.exists(fit_p) and os.path.exists(loo_p):
        FITS[name] = az.from_netcdf(fit_p); LOOD[name] = az.from_netcdf(loo_p)
        print(f"  {name:<22} loaded from disk"); continue

    t0 = time.time()
    ta = 0.90 if name in ("M0 complete pooling","M1 no pooling") else 0.99
    with build():
        ida = pm.sample(draws=DRAWS, tune=TUNE, chains=CHAINS, target_accept=ta,
                        random_seed=SEED, progressbar=True,
                        idata_kwargs={"log_likelihood": False})
    FITS[name] = ida
    thin = ida.isel(draw=slice(None, None, LL_THIN)).copy()
    m_ll = build()
    for rv in m_ll.rvs_to_initial_values: m_ll.rvs_to_initial_values[rv] = None
    with m_ll: pm.compute_log_likelihood(thin, progressbar=False)
    LOOD[name] = thin
    ida.to_netcdf(fit_p); thin.to_netcdf(loo_p)   # written immediately

    ss=ida.sample_stats; nd=int(ss.diverging.sum()); tot=CHAINS*DRAWS
    sat=float((ss.tree_depth.values>=10).mean()*100)
    flag = "" if nd==0 else ("  <- still diverging" if nd/tot>0.005 else "  <- a few")
    print(f"  {name:<22} {time.time()-t0:6.0f}s   ta {ta:.2f}   "
          f"divergences {nd:5d} ({100*nd/tot:4.1f}%)   depth10 {sat:4.1f}%{flag}")
    gc.collect()

print(f"\n{len(FITS)} models fitted here: {', '.join(FITS)}")

this notebook fits: M4 hier correlated, M7 uniform priors 



Output()

ERROR:pymc.stats.convergence:There were 2 divergences after tuning. Increase `target_accept` or reparameterize.


  M4 hier correlated       3400s   ta 0.99   divergences     2 ( 0.1%)   depth10  0.0%  <- a few


Output()

ERROR:pymc.stats.convergence:There were 16 divergences after tuning. Increase `target_accept` or reparameterize.
ERROR:pymc.stats.convergence:The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


  M7 uniform priors         840s   ta 0.99   divergences    16 ( 0.8%)   depth10  0.0%  <- still diverging

2 models fitted here: M4 hier correlated, M7 uniform priors


# Export

In [ ]:
import zipfile, os
out = "fits_worker.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(SAVE_DIR)):
        z.write(os.path.join(SAVE_DIR, f), f)
print(f"{out}  ({os.path.getsize(out)/1024**2:.0f} MB)")
print("contains:", ", ".join(sorted(os.listdir(SAVE_DIR))))
from google.colab import files
files.download(out)

fits_worker.zip  (44 MB)
contains: M4_hier_correlated_fit.nc, M4_hier_correlated_loo.nc, M7_uniform_priors_fit.nc, M7_uniform_priors_loo.nc


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>